In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


OUTPUT_DIR = Path(
    "/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs"
)

Q5_FILES = [
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2017~2018/2017_data_179_activities.csv",
        "year": 3,
        "weight_col": "wt_final_AC"
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2018~2019/2018_data_179_activities.csv",
        "year": 4,
        "weight_col": "wt_final_AC"
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/1920_london32_stable179.csv",
        "year": 5,
        "weight_col": "wt_final_online"
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/2021_london32_stable179.csv",
        "year": 6,
        "weight_col": "wt_final_online"
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year7_179activities.csv",
        "year": 7,
        "weight_col": "wt_final_online"
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year8_179activities.csv",
        "year": 8,
        "weight_col": "wt_final_online"
    },
]

REQUIRED_COLUMNS = [
    "LA_2023",
    "Age16plus",
    "Age9",
    "VolFrqB_Pop",
    "MEMS7GR_ALL",
]

MISSING_CODES = [
    -99,
    -98,
    -97,
    -96,
    -95,
    -94,
    -93,
    -92,
    -91,
]

WEIGHT_COL = "volunteering_weight"

HISTORICAL_YEARS = [
    3,
    4,
    5,
    6,
    7,
    8,
]

# Define thresholds used to flag unreliable cells.
MIN_CELL_N = 30
MIN_FREQUENT_N = 5

AGE9_LABELS = {
    2: "16-24",
    3: "25-34",
    4: "35-44",
    5: "45-54",
    6: "55-64",
    7: "65-74",
    8: "75-84",
    9: "85+",
}

VOL_BINARY_A_LABELS = {
    0: "not_twiceplus_volunteer",
    1: "twiceplus_volunteer",
}

ACTIVITY_LEVEL_LABELS = {
    0: "inactive",
    1: "fairly_active",
    2: "active",
}

AGE_GROUP_CODES = sorted(
    AGE9_LABELS.keys()
)

ACTIVITY_LEVEL_CODES = sorted(
    ACTIVITY_LEVEL_LABELS.keys()
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [2]:
def load_q5_master(file_info_list):
    yearly_dataframes = []

    for file_info in file_info_list:
        file_path = Path(file_info["path"])
        year_value = file_info["year"]
        source_weight_col = file_info["weight_col"]

        if not file_path.exists():
            raise FileNotFoundError(
                f"File not found: {file_path}"
            )

        df = pd.read_csv(
            file_path,
            low_memory=False,
        )
        
        required_columns = (
            REQUIRED_COLUMNS
            + [source_weight_col]
        )

        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns in {file_path.name}: {missing_columns}"
            )

        df = df[required_columns].copy()
        
        # Harmonise the year-specific official volunteering weight.
        df = df.rename(
            columns={
                source_weight_col: WEIGHT_COL
            }
        )
        
        df["weight_source"] = (source_weight_col)

        df["year"] = year_value

        yearly_dataframes.append(df)

    master_df = pd.concat(
        yearly_dataframes,
        ignore_index=True,
    )

    master_df = master_df.replace(
        MISSING_CODES,
        np.nan,
    )

    numeric_columns = [
        "LA_2023",
        "Age16plus",
        "Age9",
        "VolFrqB_Pop",
        "MEMS7GR_ALL",
        WEIGHT_COL,
    ]

    for column in numeric_columns:
        master_df[column] = pd.to_numeric(
            master_df[column],
            errors="coerce",
        )

    master_df = master_df.loc[
        master_df["Age16plus"] == 1
    ].copy()

    master_df["LA_2023"] = master_df[
        "LA_2023"
    ].astype("Int64")

    master_df["year"] = master_df[
        "year"
    ].astype("Int64")

    print(
        f"Done: {len(master_df):,} respondent rows, "
        f"{master_df['year'].nunique()} years, "
        f"{master_df['LA_2023'].nunique()} boroughs"
    )

    return master_df

q5_master = load_q5_master(
    file_info_list=Q5_FILES
)

print(q5_master.shape)

print("\nRows by year:")
print(
    q5_master["year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nWeight source by year:")
print(
    q5_master[
        ["year", "weight_source"]
    ]
    .drop_duplicates()
    .sort_values("year")
    .to_string(index=False)
)

print("\nNumber of boroughs:")
print(q5_master["LA_2023"].nunique())

print("\nOriginal Age9 values:")
print(
    q5_master["Age9"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nOriginal MEMS7GR_ALL values:")
print(
    q5_master["MEMS7GR_ALL"]
    .value_counts(dropna=False)
    .sort_index()
)

Done: 96,629 respondent rows, 6 years, 32 boroughs
(96629, 8)

Rows by year:
year
3    15967
4    15889
5    16091
6    16028
7    16139
8    16515
Name: count, dtype: Int64

Weight source by year:
 year   weight_source
    3     wt_final_AC
    4     wt_final_AC
    5 wt_final_online
    6 wt_final_online
    7 wt_final_online
    8 wt_final_online

Number of boroughs:
32

Original Age9 values:
Age9
2.0     7883
3.0    18022
4.0    19933
5.0    16445
6.0    14753
7.0    11772
8.0     5556
9.0     1386
NaN      879
Name: count, dtype: int64

Original MEMS7GR_ALL values:
MEMS7GR_ALL
0.0    21824
1.0    10351
2.0    64454
Name: count, dtype: int64


In [3]:
def add_q5_analysis_variables(df):
    result = df.copy()

    result["age_group_code"] = pd.to_numeric(
        result["Age9"],
        errors="coerce",
    ).astype("Int64")

    result["age_group"] = result[
        "age_group_code"
    ].map(AGE9_LABELS)

    early_years = result["year"].isin([3, 4])
    later_years = result["year"].isin([5, 6, 7, 8])

    # Create the six-year comparable volunteering variable.
    result["vol_binary_A"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )

    result.loc[
        early_years
        & (result["VolFrqB_Pop"] == 0),
        "vol_binary_A",
    ] = 0

    result.loc[
        early_years
        & (result["VolFrqB_Pop"] == 1),
        "vol_binary_A",
    ] = 1

    result.loc[
        later_years
        & result["VolFrqB_Pop"].isin([0, 1]),
        "vol_binary_A",
    ] = 0

    result.loc[
        later_years
        & result["VolFrqB_Pop"].isin([2, 3, 4]),
        "vol_binary_A",
    ] = 1

    result["vol_binary_A_label"] = result[
        "vol_binary_A"
    ].map(VOL_BINARY_A_LABELS)

    result["activity_level_3cat"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )

    valid_activity = result[
        "MEMS7GR_ALL"
    ].isin([0, 1, 2])

    result.loc[
        valid_activity,
        "activity_level_3cat",
    ] = (
        result.loc[
            valid_activity,
            "MEMS7GR_ALL",
        ]
        .astype(int)
    )

    result[
        "activity_level_3cat_label"
    ] = result["activity_level_3cat"].map(
        ACTIVITY_LEVEL_LABELS
    )

    return result

q5_master = add_q5_analysis_variables(
    q5_master
)

print("\nVersion A volunteering variable:")
print(
    q5_master["vol_binary_A"]
    .value_counts(dropna=False)
    .sort_index()
)


Version A volunteering variable:
vol_binary_A
0       64910
1        7529
<NA>    24190
Name: count, dtype: Int64


In [4]:
def make_sequential_eligibility_audit(
    df,
    weight_col,
    sample_name,
    require_age=False,
    require_activity=False,
):
    rows = []

    current_mask = pd.Series(
        True,
        index=df.index,
    )

    rows.append(
        {
            "sample_name": sample_name,
            "step": "initial_adult_rows",
            "rows_remaining": int(
                current_mask.sum()
            ),
            "excluded_at_step": 0,
        }
    )

    steps = [
        (
            "valid_volunteering",
            df["vol_binary_A"].isin(
                [0, 1]
            ),
        ),
        (
            "valid_borough",
            df["LA_2023"].notna(),
        ),
        (
            "valid_positive_weight",
            (
                df[weight_col].notna()
                & (df[weight_col] > 0)
            ),
        ),
    ]

    if require_age:
        steps.append(
            (
                "valid_age",
                df["age_group_code"].isin(
                    AGE_GROUP_CODES
                ),
            )
        )

    if require_activity:
        steps.append(
            (
                "valid_activity",
                df[
                    "activity_level_3cat"
                ].isin(
                    ACTIVITY_LEVEL_CODES
                ),
            )
        )

    previous_count = int(
        current_mask.sum()
    )

    for step_name, step_condition in steps:
        current_mask = (
            current_mask
            & step_condition
        )

        current_count = int(
            current_mask.sum()
        )

        rows.append(
            {
                "sample_name": sample_name,
                "step": step_name,
                "rows_remaining": current_count,
                "excluded_at_step": (
                    previous_count
                    - current_count
                ),
            }
        )

        previous_count = current_count

    audit = pd.DataFrame(
        rows
    )

    audit[
        "percentage_of_initial"
    ] = (
        audit[
            "rows_remaining"
        ]
        / len(df)
        * 100
    )

    return audit

weight_status_summary = pd.DataFrame(
    {
        "weight_status": [
            "missing",
            "zero",
            "negative",
            "positive",
        ],
        "rows": [
            q5_master[
                WEIGHT_COL
            ].isna().sum(),
            (
                q5_master[
                    WEIGHT_COL
                ]
                == 0
            ).sum(),
            (
                q5_master[
                    WEIGHT_COL
                ]
                < 0
            ).sum(),
            (
                q5_master[
                    WEIGHT_COL
                ]
                > 0
            ).sum(),
        ],
    }
)

weight_status_summary[
    "percentage_of_master"
] = (
    weight_status_summary[
        "rows"
    ]
    / len(q5_master)
    * 100
)

print(
    "Survey-weight status:"
)
display(
    weight_status_summary
)

borough_age_activity_audit = (
    make_sequential_eligibility_audit(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name=(
            "borough_age_activity"
        ),
        require_age=True,
        require_activity=True,
    )
)

borough_age_audit = (
    make_sequential_eligibility_audit(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name="borough_age",
        require_age=True,
        require_activity=False,
    )
)

borough_activity_audit = (
    make_sequential_eligibility_audit(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name="borough_activity",
        require_age=False,
        require_activity=True,
    )
)

eligibility_audit = pd.concat(
    [
        borough_age_activity_audit,
        borough_age_audit,
        borough_activity_audit,
    ],
    ignore_index=True,
)

print(
    "\nSequential eligibility audit:"
)

display(
    eligibility_audit
)

Survey-weight status:


,weight_status,rows,percentage_of_master
0,missing,26373,27.293049
1,zero,0,0.000000
2,negative,0,0.000000
3,positive,70256,72.706951



Sequential eligibility audit:


,sample_name,step,rows_remaining,excluded_at_step,percentage_of_initial
0,borough_age_activity,initial_adult_rows,96629,0,100.000000
1,borough_age_activity,valid_volunteering,72439,24190,74.966107
2,borough_age_activity,valid_borough,72439,0,74.966107
3,borough_age_activity,valid_positive_weight,69065,3374,71.474402
4,borough_age_activity,valid_age,68630,435,71.024227
5,borough_age_activity,valid_activity,68630,0,71.024227
6,borough_age,initial_adult_rows,96629,0,100.000000
7,borough_age,valid_volunteering,72439,24190,74.966107
8,borough_age,valid_borough,72439,0,74.966107
9,borough_age,valid_positive_weight,69065,3374,71.474402


In [5]:
def prepare_q5_analysis_sample(
    df,
    weight_col,
    sample_name,
    require_age=False,
    require_activity=False,
):
    result = df.copy()

    eligible = (
        result["year"].isin(
            HISTORICAL_YEARS
        )
        & result["LA_2023"].notna()
        & result["vol_binary_A"].isin([0, 1])
        & result[weight_col].notna()
        & (result[weight_col] > 0)
    )

    if require_age:
        eligible = (
            eligible
            & result["age_group_code"].isin(
                AGE_GROUP_CODES
            )
        )

    if require_activity:
        eligible = (
            eligible
            & result[
                "activity_level_3cat"
            ].isin(
                ACTIVITY_LEVEL_CODES
            )
        )

    analysis_sample = result.loc[
        eligible
    ].copy()

    analysis_sample["frequent_case"] = (
        analysis_sample["vol_binary_A"] == 1
    ).astype(int)

    analysis_sample["not_frequent_case"] = (
        analysis_sample["vol_binary_A"] == 0
    ).astype(int)

    analysis_sample["weighted_frequent"] = (
        analysis_sample[weight_col]
        * analysis_sample["frequent_case"]
    )

    analysis_sample[
        "weighted_not_frequent"
    ] = (
        analysis_sample[weight_col]
        * analysis_sample[
            "not_frequent_case"
        ]
    )

    analysis_sample["weight_squared"] = (
        analysis_sample[weight_col] ** 2
    )

    print(
        f"{sample_name} eligible respondent rows: "
        f"{len(analysis_sample):,}"
    )

    print(
        f"{sample_name} excluded respondent rows: "
        f"{len(result) - len(analysis_sample):,}"
    )

    return analysis_sample


q5_borough_age_activity_sample = (
    prepare_q5_analysis_sample(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name=(
            "Borough-age-activity"
        ),
        require_age=True,
        require_activity=True,
    )
)

q5_borough_age_sample = (
    prepare_q5_analysis_sample(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name="Borough-age",
        require_age=True,
        require_activity=False,
    )
)

q5_borough_activity_sample = (
    prepare_q5_analysis_sample(
        df=q5_master,
        weight_col=WEIGHT_COL,
        sample_name="Borough-activity",
        require_age=False,
        require_activity=True,
    )
)

analysis_sample_summary = pd.DataFrame(
    {
        "panel_level": [
            "borough_age_activity",
            "borough_age",
            "borough_activity",
        ],
        "eligible_respondent_rows": [
            len(
                q5_borough_age_activity_sample
            ),
            len(
                q5_borough_age_sample
            ),
            len(
                q5_borough_activity_sample
            ),
        ],
    }
)

analysis_sample_summary[
    "additional_rows_vs_joint_sample"
] = (
    analysis_sample_summary[
        "eligible_respondent_rows"
    ]
    - len(
        q5_borough_age_activity_sample
    )
)

print("\nAnalysis sample sizes:")
display(
    analysis_sample_summary
)

print(
    "\nBorough-age-activity eligible rows by year:"
)
print(
    q5_borough_age_activity_sample[
        "year"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nBorough-age eligible rows by year:"
)
print(
    q5_borough_age_sample[
        "year"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nBorough-activity eligible rows by year:"
)
print(
    q5_borough_activity_sample[
        "year"
    ]
    .value_counts()
    .sort_index()
)

Borough-age-activity eligible respondent rows: 68,630
Borough-age-activity excluded respondent rows: 27,999
Borough-age eligible respondent rows: 68,630
Borough-age excluded respondent rows: 27,999
Borough-activity eligible respondent rows: 69,065
Borough-activity excluded respondent rows: 27,564

Analysis sample sizes:


,panel_level,eligible_respondent_rows,additional_rows_vs_joint_sample
0,borough_age_activity,68630,0
1,borough_age,68630,0
2,borough_activity,69065,435



Borough-age-activity eligible rows by year:
year
3    10230
4    10087
5    11677
6    11930
7    12081
8    12625
Name: count, dtype: Int64

Borough-age eligible rows by year:
year
3    10230
4    10087
5    11677
6    11930
7    12081
8    12625
Name: count, dtype: Int64

Borough-activity eligible rows by year:
year
3    10318
4    10206
5    11719
6    11988
7    12144
8    12690
Name: count, dtype: Int64


In [6]:
def make_q5_observed_panel(
    analysis_sample,
    group_columns,
    weight_col,
    panel_name,
    min_cell_n=30,
    min_frequent_n=5,
):
    observed_panel = (
        analysis_sample
        .groupby(
            group_columns,
            dropna=False,
            observed=True,
        )
        .agg(
            n_parent_cell=(
                "vol_binary_A",
                "size",
            ),
            weighted_n_parent_cell=(
                weight_col,
                "sum",
            ),
            sum_weight_squared=(
                "weight_squared",
                "sum",
            ),
            n_frequent_volunteer=(
                "frequent_case",
                "sum",
            ),
            weighted_n_frequent_volunteer=(
                "weighted_frequent",
                "sum",
            ),
            n_not_frequent_volunteer=(
                "not_frequent_case",
                "sum",
            ),
            weighted_n_not_frequent_volunteer=(
                "weighted_not_frequent",
                "sum",
            ),
        )
        .reset_index()
    )

    # Calculate the weighted twice-plus volunteering rate.
    observed_panel[
        "frequent_volunteer_rate"
    ] = (
        observed_panel[
            "weighted_n_frequent_volunteer"
        ]
        / observed_panel[
            "weighted_n_parent_cell"
        ]
    )

    # Calculate the complementary not-twice-plus rate.
    observed_panel[
        "not_frequent_volunteer_rate"
    ] = (
        observed_panel[
            "weighted_n_not_frequent_volunteer"
        ]
        / observed_panel[
            "weighted_n_parent_cell"
        ]
    )

    # Calculate the effective sample size from the survey weights.
    observed_panel[
        "effective_n_parent_cell"
    ] = (
        observed_panel[
            "weighted_n_parent_cell"
        ] ** 2
        / observed_panel[
            "sum_weight_squared"
        ]
    )

    if (
        "age_group_code"
        in observed_panel.columns
    ):
        observed_panel[
            "age_group"
        ] = observed_panel[
            "age_group_code"
        ].map(
            AGE9_LABELS
        )

    if (
        "activity_level_3cat"
        in observed_panel.columns
    ):
        observed_panel[
            "activity_level_3cat_label"
        ] = observed_panel[
            "activity_level_3cat"
        ].map(
            ACTIVITY_LEVEL_LABELS
        )

    observed_panel["small_cell"] = (
        observed_panel[
            "n_parent_cell"
        ]
        < min_cell_n
    )

    observed_panel[
        "few_frequent_cases"
    ] = (
        observed_panel[
            "n_frequent_volunteer"
        ]
        < min_frequent_n
    )

    observed_panel[
        "zero_frequent_cases"
    ] = (
        observed_panel[
            "n_frequent_volunteer"
        ]
        == 0
    )

    observed_panel["cell_observed"] = True

    observed_panel = (
        observed_panel
        .sort_values(
            group_columns
        )
        .reset_index(drop=True)
    )

    print(
        f"Done: {len(observed_panel):,} "
        f"observed {panel_name} panel rows"
    )

    return observed_panel


q5_borough_age_activity_observed_panel = (
    make_q5_observed_panel(
        analysis_sample=(
            q5_borough_age_activity_sample
        ),
        group_columns=[
            "year",
            "LA_2023",
            "age_group_code",
            "activity_level_3cat",
        ],
        weight_col=WEIGHT_COL,
        panel_name="borough-age-activity",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

q5_borough_age_observed_panel = (
    make_q5_observed_panel(
        analysis_sample=(
            q5_borough_age_sample
        ),
        group_columns=[
            "year",
            "LA_2023",
            "age_group_code",
        ],
        weight_col=WEIGHT_COL,
        panel_name="borough-age",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

q5_borough_activity_observed_panel = (
    make_q5_observed_panel(
        analysis_sample=(
            q5_borough_activity_sample
        ),
        group_columns=[
            "year",
            "LA_2023",
            "activity_level_3cat",
        ],
        weight_col=WEIGHT_COL,
        panel_name="borough-activity",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

print(
    "\nObserved borough-age-activity cells by year:"
)
print(
    q5_borough_age_activity_observed_panel[
        "year"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nObserved borough-age cells by year:"
)
print(
    q5_borough_age_observed_panel[
        "year"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nObserved borough-activity cells by year:"
)
print(
    q5_borough_activity_observed_panel[
        "year"
    ]
    .value_counts()
    .sort_index()
)

Done: 4,290 observed borough-age-activity panel rows
Done: 1,523 observed borough-age panel rows
Done: 576 observed borough-activity panel rows

Observed borough-age-activity cells by year:
year
3    734
4    728
5    695
6    704
7    713
8    716
Name: count, dtype: Int64

Observed borough-age cells by year:
year
3    256
4    254
5    250
6    254
7    254
8    255
Name: count, dtype: Int64

Observed borough-activity cells by year:
year
3    96
4    96
5    96
6    96
7    96
8    96
Name: count, dtype: Int64


In [7]:
def complete_q5_panel(
    observed_panel,
    master_df,
    dimension_columns,
    panel_name,
    min_cell_n=30,
    min_frequent_n=5,
):
    boroughs = sorted(
        master_df["LA_2023"]
        .dropna()
        .astype(int)
        .unique()
    )

    dimension_values = []

    for column in dimension_columns:
        if column == "age_group_code":
            dimension_values.append(
                AGE_GROUP_CODES
            )

        elif column == "activity_level_3cat":
            dimension_values.append(
                ACTIVITY_LEVEL_CODES
            )

        else:
            raise ValueError(
                f"Unknown panel dimension: "
                f"{column}"
            )

    # Create every theoretical year-borough-dimension combination.
    complete_index = pd.MultiIndex.from_product(
        [
            HISTORICAL_YEARS,
            boroughs,
            *dimension_values,
        ],
        names=[
            "year",
            "LA_2023",
            *dimension_columns,
        ],
    )

    complete_grid = (
        complete_index
        .to_frame(index=False)
    )

    key_columns = [
        "year",
        "LA_2023",
        *dimension_columns,
    ]

    measure_columns = [
        "frequent_volunteer_rate",
        "not_frequent_volunteer_rate",
        "n_parent_cell",
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "effective_n_parent_cell",
        "n_frequent_volunteer",
        "weighted_n_frequent_volunteer",
        "n_not_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
        "cell_observed",
    ]

    complete_panel = complete_grid.merge(
        observed_panel[
            key_columns
            + measure_columns
        ],
        on=key_columns,
        how="left",
        validate="one_to_one",
    )

    # Add readable category labels.
    if (
        "age_group_code"
        in dimension_columns
    ):
        complete_panel[
            "age_group"
        ] = complete_panel[
            "age_group_code"
        ].map(
            AGE9_LABELS
        )

    if (
        "activity_level_3cat"
        in dimension_columns
    ):
        complete_panel[
            "activity_level_3cat_label"
        ] = complete_panel[
            "activity_level_3cat"
        ].map(
            ACTIVITY_LEVEL_LABELS
        )

    # Convert the merged indicator before filling missing values.
    complete_panel["cell_observed"] = (
        complete_panel["cell_observed"]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

    raw_count_columns = [
        "n_parent_cell",
        "n_frequent_volunteer",
        "n_not_frequent_volunteer",
    ]

    weighted_count_columns = [
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "weighted_n_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
    ]

    # An unobserved cell contains zero eligible respondents.
    for column in raw_count_columns:
        complete_panel[column] = (
            complete_panel[column]
            .fillna(0)
            .astype(int)
        )

    # Weighted counts are zero when no eligible respondent exists.
    for column in weighted_count_columns:
        complete_panel[column] = (
            complete_panel[column]
            .fillna(0.0)
        )

    # Effective sample size is undefined when no respondent exists.
    complete_panel.loc[
        ~complete_panel["cell_observed"],
        "effective_n_parent_cell",
    ] = np.nan

    complete_panel["rate_available"] = (
        complete_panel["cell_observed"]
        & complete_panel[
            "frequent_volunteer_rate"
        ].notna()
    )

    observed_mask = complete_panel[
        "cell_observed"
    ]

    # Reliability flags are only defined for observed cells.
    complete_panel["small_cell"] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "small_cell",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_parent_cell",
        ]
        < min_cell_n
    )

    complete_panel[
        "few_frequent_cases"
    ] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "few_frequent_cases",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_frequent_volunteer",
        ]
        < min_frequent_n
    )

    complete_panel[
        "zero_frequent_cases"
    ] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "zero_frequent_cases",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_frequent_volunteer",
        ]
        == 0
    )

    # Create one identifier for each time series.
    complete_panel["series_id"] = (
        "borough_"
        + complete_panel[
            "LA_2023"
        ].astype(str)
    )

    if (
        "age_group_code"
        in dimension_columns
    ):
        complete_panel["series_id"] = (
            complete_panel["series_id"]
            + "__age_"
            + complete_panel[
                "age_group_code"
            ].astype(str)
        )

    if (
        "activity_level_3cat"
        in dimension_columns
    ):
        complete_panel["series_id"] = (
            complete_panel["series_id"]
            + "__activity_"
            + complete_panel[
                "activity_level_3cat"
            ].astype(str)
        )

    complete_panel[
        "observed_years_in_series"
    ] = (
        complete_panel
        .groupby("series_id")[
            "cell_observed"
        ]
        .transform("sum")
        .astype(int)
    )

    complete_panel[
        "series_coverage_rate"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        / len(
            HISTORICAL_YEARS
        )
    )

    # Identify series with at least one observed historical year.
    complete_panel[
        "series_has_history"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        > 0
    )

    # Identify series with all six historical years observed.
    complete_panel[
        "series_complete_history"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        == len(
            HISTORICAL_YEARS
        )
    )

    dimension_output_columns = []

    if (
        "age_group_code"
        in dimension_columns
    ):
        dimension_output_columns.extend(
            [
                "age_group_code",
                "age_group",
            ]
        )

    if (
        "activity_level_3cat"
        in dimension_columns
    ):
        dimension_output_columns.extend(
            [
                "activity_level_3cat",
                "activity_level_3cat_label",
            ]
        )

    final_columns = [
        "series_id",
        "year",
        "LA_2023",
        *dimension_output_columns,
        "frequent_volunteer_rate",
        "not_frequent_volunteer_rate",
        "n_parent_cell",
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "effective_n_parent_cell",
        "n_frequent_volunteer",
        "weighted_n_frequent_volunteer",
        "n_not_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
        "small_cell",
        "few_frequent_cases",
        "zero_frequent_cases",
        "cell_observed",
        "rate_available",
        "observed_years_in_series",
        "series_coverage_rate",
        "series_has_history",
        "series_complete_history",
    ]

    sort_columns = [
        "LA_2023",
        *dimension_columns,
        "year",
    ]

    complete_panel = (
        complete_panel[
            final_columns
        ]
        .sort_values(
            sort_columns
        )
        .reset_index(drop=True)
    )

    print(
        f"Done: {len(complete_panel):,} rows in the complete {panel_name} panel"
    )

    print(
        f"Observed cells: {complete_panel['cell_observed'].sum():,}"
    )

    print(
        f"Unobserved cells: {(~complete_panel['cell_observed']).sum():,}"
    )

    return complete_panel


q5_borough_age_activity_complete_panel = (
    complete_q5_panel(
        observed_panel=(
            q5_borough_age_activity_observed_panel
        ),
        master_df=q5_master,
        dimension_columns=[
            "age_group_code",
            "activity_level_3cat",
        ],
        panel_name="borough-age-activity",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

q5_borough_age_complete_panel = (
    complete_q5_panel(
        observed_panel=(
            q5_borough_age_observed_panel
        ),
        master_df=q5_master,
        dimension_columns=[
            "age_group_code",
        ],
        panel_name="borough-age",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

q5_borough_activity_complete_panel = (
    complete_q5_panel(
        observed_panel=(
            q5_borough_activity_observed_panel
        ),
        master_df=q5_master,
        dimension_columns=[
            "activity_level_3cat",
        ],
        panel_name="borough-activity",
        min_cell_n=MIN_CELL_N,
        min_frequent_n=MIN_FREQUENT_N,
    )
)

Done: 4,608 rows in the complete borough-age-activity panel
Observed cells: 4,290
Unobserved cells: 318
Done: 1,536 rows in the complete borough-age panel
Observed cells: 1,523
Unobserved cells: 13
Done: 576 rows in the complete borough-activity panel
Observed cells: 576
Unobserved cells: 0


In [8]:
def validate_q5_panel(
    panel,
    dimension_columns,
    panel_name,
):
    key_columns = [
        "year",
        "LA_2023",
        *dimension_columns,
    ]

    reliability_flag_columns = [
        "small_cell",
        "few_frequent_cases",
        "zero_frequent_cases",
    ]

    expected_rows = (
        len(HISTORICAL_YEARS)
        * 32
    )

    if (
        "age_group_code"
        in dimension_columns
    ):
        expected_rows *= len(
            AGE_GROUP_CODES
        )

    if (
        "activity_level_3cat"
        in dimension_columns
    ):
        expected_rows *= len(
            ACTIVITY_LEVEL_CODES
        )

    # Check whether the panel key is unique.
    duplicate_count = panel.duplicated(
        subset=key_columns
    ).sum()

    if duplicate_count != 0:
        raise ValueError(
            f"Duplicate panel keys found: {duplicate_count}"
        )

    # Check whether the complete panel contains all theoretical combinations.
    if len(panel) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} panel rows, but found {len(panel)}"
        )

    # Check whether all six historical years are included.
    if sorted(
        panel["year"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    ) != HISTORICAL_YEARS:
        raise ValueError(
            "The final dataset does not contain the expected six historical years"
        )

    # Check whether all 32 London boroughs are included.
    if panel["LA_2023"].nunique() != 32:
        raise ValueError(
            "The final dataset does not contain exactly 32 boroughs"
        )

    if (
        "age_group_code"
        in dimension_columns
        and panel[
            "age_group_code"
        ].nunique()
        != len(
            AGE_GROUP_CODES
        )
    ):
        raise ValueError(
            "The final dataset does not contain all eight age groups"
        )

    if (
        "activity_level_3cat"
        in dimension_columns
        and panel[
            "activity_level_3cat"
        ].nunique()
        != len(
            ACTIVITY_LEVEL_CODES
        )
    ):
        raise ValueError(
            "The final dataset does not contain all three activity levels"
        )

    # Separate observed and unobserved cells.
    observed = panel.loc[
        panel["cell_observed"]
    ].copy()

    unobserved = panel.loc[
        ~panel["cell_observed"]
    ].copy()

    # Check that all observed cells contain both volunteering rates.
    if observed[
        "frequent_volunteer_rate"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing frequent volunteer rate"
        )

    if observed[
        "not_frequent_volunteer_rate"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing not-frequent volunteer rate"
        )

    # Check that all observed rates fall within the valid range.
    if not observed[
        "frequent_volunteer_rate"
    ].between(0, 1).all():
        raise ValueError(
            "Some frequent volunteer rates fall outside the 0-1 range"
        )

    if not observed[
        "not_frequent_volunteer_rate"
    ].between(0, 1).all():
        raise ValueError(
            "Some not-frequent volunteer rates fall outside the 0-1 range"
        )

    # Check that the two volunteering rates sum to one.
    rate_sum = (
        observed[
            "frequent_volunteer_rate"
        ]
        + observed[
            "not_frequent_volunteer_rate"
        ]
    )

    if not np.allclose(
        rate_sum,
        1.0,
        atol=1e-10,
    ):
        raise ValueError(
            "The two volunteering rates do not sum to one"
        )

    # Check that the two raw category counts equal the parent-cell count.
    raw_count_sum = (
        observed[
            "n_frequent_volunteer"
        ]
        + observed[
            "n_not_frequent_volunteer"
        ]
    )

    if not (
        raw_count_sum
        == observed[
            "n_parent_cell"
        ]
    ).all():
        raise ValueError(
            "Raw category counts do not equal the parent-cell count"
        )

    # Check that the two weighted category counts equal the weighted total.
    weighted_count_sum = (
        observed[
            "weighted_n_frequent_volunteer"
        ]
        + observed[
            "weighted_n_not_frequent_volunteer"
        ]
    )

    if not np.allclose(
        weighted_count_sum,
        observed[
            "weighted_n_parent_cell"
        ],
        atol=1e-10,
    ):
        raise ValueError(
            "Weighted category counts do not equal the weighted parent-cell count"
        )

    # Check that observed cells contain a valid effective sample size.
    if observed[
        "effective_n_parent_cell"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing effective sample size"
        )

    if not (
        observed[
            "effective_n_parent_cell"
        ]
        > 0
    ).all():
        raise ValueError(
            "Observed effective sample sizes must be greater than zero"
        )

    # Effective sample size should not exceed the raw sample size.
    if not (
        observed[
            "effective_n_parent_cell"
        ]
        <= observed[
            "n_parent_cell"
        ]
        + 1e-10
    ).all():
        raise ValueError(
            "Effective sample size cannot exceed the raw cell sample size"
        )

    # Check that all observed cells contain reliability indicators.
    if observed[
        reliability_flag_columns
    ].isna().any().any():
        raise ValueError(
            "Observed cells must contain all reliability flags"
        )

    # Check that unobserved cells do not contain estimated rates.
    if unobserved[
        "frequent_volunteer_rate"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain a frequent volunteer rate"
        )

    if unobserved[
        "not_frequent_volunteer_rate"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain a not-frequent volunteer rate"
        )

    # Check that effective sample size remains missing for unobserved cells.
    if unobserved[
        "effective_n_parent_cell"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain an effective sample size"
        )

    # Check that reliability indicators remain missing for unobserved cells.
    if unobserved[
        reliability_flag_columns
    ].notna().any().any():
        raise ValueError(
            "Reliability flags must remain missing for unobserved cells"
        )

    # Check whether the rate availability indicator is consistent.
    expected_rate_available = (
        panel["cell_observed"]
        & panel[
            "frequent_volunteer_rate"
        ].notna()
    )

    if not (
        panel["rate_available"]
        == expected_rate_available
    ).all():
        raise ValueError(
            "The rate_available indicator is inconsistent"
        )

    # Check that every series contains exactly six yearly rows.
    series_lengths = (
        panel
        .groupby("series_id")
        .size()
    )

    if not (
        series_lengths
        == len(
            HISTORICAL_YEARS
        )
    ).all():
        raise ValueError(
            "Every series must contain exactly six yearly rows"
        )

    # Recalculate the number of observed years in each series.
    calculated_observed_years = (
        panel
        .groupby(
            "series_id"
        )["cell_observed"]
        .transform("sum")
        .astype(int)
    )

    if not (
        calculated_observed_years
        == panel[
            "observed_years_in_series"
        ]
    ).all():
        raise ValueError(
            "The series coverage count is inconsistent"
        )

    # Recalculate and validate the series coverage rate.
    calculated_coverage_rate = (
        calculated_observed_years
        / len(
            HISTORICAL_YEARS
        )
    )

    if not np.allclose(
        calculated_coverage_rate,
        panel[
            "series_coverage_rate"
        ],
        atol=1e-10,
    ):
        raise ValueError(
            "The series coverage rate is inconsistent"
        )

    complete_series_count = panel.loc[
        panel[
            "series_complete_history"
        ],
        "series_id",
    ].nunique()

    total_series_count = (
        panel[
            "series_id"
        ].nunique()
    )

    print(
        f"All {panel_name} panel validation checks passed."
    )

    print(
        f"Rows: {len(panel):,}"
    )

    print(
        f"Boroughs: {panel['LA_2023'].nunique()}"
    )

    print(
        f"Observed cells: {panel['cell_observed'].sum():,}"
    )

    print(
        f"Unobserved cells: {(~panel['cell_observed']).sum():,}"
    )

    print(
        f"Total series: {total_series_count:,}"
    )

    print(
        f"Complete six-year series: {complete_series_count:,}"
    )


validate_q5_panel(
    panel=(
        q5_borough_age_activity_complete_panel
    ),
    dimension_columns=[
        "age_group_code",
        "activity_level_3cat",
    ],
    panel_name="borough-age-activity",
)

print()

validate_q5_panel(
    panel=(
        q5_borough_age_complete_panel
    ),
    dimension_columns=[
        "age_group_code",
    ],
    panel_name="borough-age",
)

print()

validate_q5_panel(
    panel=(
        q5_borough_activity_complete_panel
    ),
    dimension_columns=[
        "activity_level_3cat",
    ],
    panel_name="borough-activity",
)

All borough-age-activity panel validation checks passed.
Rows: 4,608
Boroughs: 32
Observed cells: 4,290
Unobserved cells: 318
Total series: 768
Complete six-year series: 632

All borough-age panel validation checks passed.
Rows: 1,536
Boroughs: 32
Observed cells: 1,523
Unobserved cells: 13
Total series: 256
Complete six-year series: 246

All borough-activity panel validation checks passed.
Rows: 576
Boroughs: 32
Observed cells: 576
Unobserved cells: 0
Total series: 96
Complete six-year series: 96


In [9]:
def summarise_panel_quality(
    panel,
    panel_level,
):
    observed = panel.loc[
        panel["cell_observed"]
    ].copy()

    return {
        "panel_level": panel_level,
        "total_panel_rows": len(
            panel
        ),
        "observed_cells": len(
            observed
        ),
        "observed_cell_rate": (
            len(observed)
            / len(panel)
            * 100
        ),
        "median_raw_n": observed[
            "n_parent_cell"
        ].median(),
        "median_effective_n": observed[
            "effective_n_parent_cell"
        ].median(),
        "small_cell_rate": (
            observed[
                "small_cell"
            ].mean()
            * 100
        ),
        "few_frequent_cases_rate": (
            observed[
                "few_frequent_cases"
            ].mean()
            * 100
        ),
        "zero_frequent_cases_rate": (
            observed[
                "zero_frequent_cases"
            ].mean()
            * 100
        ),
        "effective_n_below_5_rate": (
            (
                observed[
                    "effective_n_parent_cell"
                ]
                < 5
            ).mean()
            * 100
        ),
        "effective_n_below_10_rate": (
            (
                observed[
                    "effective_n_parent_cell"
                ]
                < 10
            ).mean()
            * 100
        ),
    }


panel_quality_comparison = pd.DataFrame(
    [
        summarise_panel_quality(
            panel=(
                q5_borough_age_activity_complete_panel
            ),
            panel_level=(
                "borough_age_activity"
            ),
        ),
        summarise_panel_quality(
            panel=(
                q5_borough_age_complete_panel
            ),
            panel_level=(
                "borough_age"
            ),
        ),
        summarise_panel_quality(
            panel=(
                q5_borough_activity_complete_panel
            ),
            panel_level=(
                "borough_activity"
            ),
        ),
    ]
)

print(
    "Overall panel quality comparison:"
)
display(
    panel_quality_comparison
)

def summarise_quality_by_year(
    panel,
    panel_level,
):
    observed = panel.loc[
        panel["cell_observed"]
    ].copy()

    observed[
        "effective_n_below_5"
    ] = (
        observed[
            "effective_n_parent_cell"
        ]
        < 5
    )

    observed[
        "effective_n_below_10"
    ] = (
        observed[
            "effective_n_parent_cell"
        ]
        < 10
    )

    summary = (
        observed
        .groupby(
            "year",
            observed=True,
        )
        .agg(
            observed_cells=(
                "series_id",
                "size",
            ),
            median_raw_n=(
                "n_parent_cell",
                "median",
            ),
            median_effective_n=(
                "effective_n_parent_cell",
                "median",
            ),
            small_cell_rate=(
                "small_cell",
                "mean",
            ),
            few_frequent_cases_rate=(
                "few_frequent_cases",
                "mean",
            ),
            zero_frequent_cases_rate=(
                "zero_frequent_cases",
                "mean",
            ),
            effective_n_below_5_rate=(
                "effective_n_below_5",
                "mean",
            ),
            effective_n_below_10_rate=(
                "effective_n_below_10",
                "mean",
            ),
        )
        .reset_index()
    )

    percentage_columns = [
        "small_cell_rate",
        "few_frequent_cases_rate",
        "zero_frequent_cases_rate",
        "effective_n_below_5_rate",
        "effective_n_below_10_rate",
    ]

    summary[
        percentage_columns
    ] = (
        summary[
            percentage_columns
        ]
        * 100
    )

    summary.insert(
        0,
        "panel_level",
        panel_level,
    )

    return summary


year_quality_summary = pd.concat(
    [
        summarise_quality_by_year(
            panel=(
                q5_borough_age_activity_complete_panel
            ),
            panel_level=(
                "borough_age_activity"
            ),
        ),
        summarise_quality_by_year(
            panel=(
                q5_borough_age_complete_panel
            ),
            panel_level=(
                "borough_age"
            ),
        ),
        summarise_quality_by_year(
            panel=(
                q5_borough_activity_complete_panel
            ),
            panel_level=(
                "borough_activity"
            ),
        ),
    ],
    ignore_index=True,
)

print(
    "\nCell quality by panel and year:"
)
display(
    year_quality_summary
)

def make_series_coverage_summary(
    panel,
    panel_level,
):
    series_quality = (
        panel[
            [
                "series_id",
                "observed_years_in_series",
                "series_coverage_rate",
                "series_has_history",
                "series_complete_history",
            ]
        ]
        .drop_duplicates(
            subset="series_id"
        )
    )

    summary = (
        series_quality
        .groupby(
            "observed_years_in_series",
            observed=True,
        )
        .size()
        .rename(
            "number_of_series"
        )
        .reset_index()
        .sort_values(
            "observed_years_in_series"
        )
    )

    summary[
        "percentage_of_series"
    ] = (
        summary[
            "number_of_series"
        ]
        / len(
            series_quality
        )
        * 100
    )

    summary.insert(
        0,
        "panel_level",
        panel_level,
    )

    return summary


series_coverage_summary = pd.concat(
    [
        make_series_coverage_summary(
            panel=(
                q5_borough_age_activity_complete_panel
            ),
            panel_level=(
                "borough_age_activity"
            ),
        ),
        make_series_coverage_summary(
            panel=(
                q5_borough_age_complete_panel
            ),
            panel_level=(
                "borough_age"
            ),
        ),
        make_series_coverage_summary(
            panel=(
                q5_borough_activity_complete_panel
            ),
            panel_level=(
                "borough_activity"
            ),
        ),
    ],
    ignore_index=True,
)

print(
    "\nHistorical coverage by panel:"
)
display(
    series_coverage_summary
)


Overall panel quality comparison:


,panel_level,total_panel_rows,observed_cells,observed_cell_rate,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate
0,borough_age_activity,4608,4290,93.098958,9.0,6.371735,81.468531,87.016317,46.829837,39.836830,66.573427
1,borough_age,1536,1523,99.153646,45.0,27.614169,33.420880,52.133946,15.036113,11.424819,18.384767
2,borough_activity,576,576,100.000000,76.0,45.759476,3.298611,45.659722,2.951389,0.000000,0.000000



Cell quality by panel and year:


,panel_level,year,observed_cells,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate
0,borough_age_activity,3,734,8.0,5.602590,83.787466,89.237057,54.359673,44.414169,71.253406
1,borough_age_activity,4,728,8.0,5.546044,83.653846,90.934066,53.021978,45.054945,73.489011
2,borough_age_activity,5,695,9.0,6.291137,80.431655,84.604317,38.273381,36.834532,65.467626
3,borough_age_activity,6,704,10.0,7.046294,80.681818,89.488636,48.4375,38.920455,63.494318
4,borough_age_activity,7,713,10.0,6.986699,80.785414,85.69425,44.460028,36.465638,64.656381
5,borough_age_activity,8,716,10.0,6.970184,79.329609,81.98324,41.899441,37.011173,60.754190
6,borough_age,3,256,40.5,23.984157,36.71875,62.109375,17.578125,7.421875,17.187500
7,borough_age,4,254,41.5,24.114212,35.826772,64.173228,13.385827,9.448819,18.897638
8,borough_age,5,250,46.0,27.973078,31.2,44.4,12.0,12.800000,18.400000
9,borough_age,6,254,46.0,29.644053,32.283465,55.511811,16.141732,13.385827,20.078740



Historical coverage by panel:


,panel_level,observed_years_in_series,number_of_series,percentage_of_series
0,borough_age_activity,0,1,0.130208
1,borough_age_activity,1,16,2.083333
2,borough_age_activity,2,12,1.562500
3,borough_age_activity,3,22,2.864583
4,borough_age_activity,4,33,4.296875
5,borough_age_activity,5,52,6.770833
6,borough_age_activity,6,632,82.291667
7,borough_age,4,3,1.171875
8,borough_age,5,7,2.734375
9,borough_age,6,246,96.093750


In [10]:
respondent_output_path = (
    OUTPUT_DIR
    / "q5_respondent_master_with_variables.csv"
)

borough_age_activity_observed_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_activitylevel_panel_observed.csv"
)

borough_age_activity_complete_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_activitylevel_panel_complete.csv"
)

borough_age_observed_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_panel_observed.csv"
)

borough_age_complete_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_panel_complete.csv"
)

borough_activity_observed_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_activitylevel_panel_observed.csv"
)

borough_activity_complete_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_activitylevel_panel_complete.csv"
)

sample_summary_output_path = (
    OUTPUT_DIR
    / "q5_analysis_sample_summary.csv"
)

panel_quality_output_path = (
    OUTPUT_DIR
    / "q5_panel_quality_comparison.csv"
)

year_quality_output_path = (
    OUTPUT_DIR
    / "q5_panel_quality_by_year.csv"
)

series_coverage_output_path = (
    OUTPUT_DIR
    / "q5_series_coverage_summary.csv"
)


def prepare_observed_panel_for_export(
    complete_panel,
):
    observed_panel = (
        complete_panel.loc[
            complete_panel[
                "cell_observed"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    reliability_flag_columns = [
        "small_cell",
        "few_frequent_cases",
        "zero_frequent_cases",
    ]

    for column in reliability_flag_columns:
        observed_panel[column] = (
            observed_panel[column]
            .astype(bool)
        )

    return observed_panel


q5_borough_age_activity_observed_export = (
    prepare_observed_panel_for_export(
        q5_borough_age_activity_complete_panel
    )
)

q5_borough_age_observed_export = (
    prepare_observed_panel_for_export(
        q5_borough_age_complete_panel
    )
)

q5_borough_activity_observed_export = (
    prepare_observed_panel_for_export(
        q5_borough_activity_complete_panel
    )
)

q5_master.to_csv(
    respondent_output_path,
    index=False,
)

q5_borough_age_activity_observed_export.to_csv(
    borough_age_activity_observed_path,
    index=False,
)

q5_borough_age_activity_complete_panel.to_csv(
    borough_age_activity_complete_path,
    index=False,
)

q5_borough_age_observed_export.to_csv(
    borough_age_observed_path,
    index=False,
)

q5_borough_age_complete_panel.to_csv(
    borough_age_complete_path,
    index=False,
)

q5_borough_activity_observed_export.to_csv(
    borough_activity_observed_path,
    index=False,
)

q5_borough_activity_complete_panel.to_csv(
    borough_activity_complete_path,
    index=False,
)

analysis_sample_summary.to_csv(
    sample_summary_output_path,
    index=False,
)

panel_quality_comparison.to_csv(
    panel_quality_output_path,
    index=False,
)

year_quality_summary.to_csv(
    year_quality_output_path,
    index=False,
)

series_coverage_summary.to_csv(
    series_coverage_output_path,
    index=False,
)

print(
    "Saved respondent-level dataset:"
)
print(
    respondent_output_path
)
print(
    f"Rows: {len(q5_master):,}"
)

print(
    "\nSaved borough-age-activity observed panel:"
)
print(
    borough_age_activity_observed_path
)
print(
    f"Rows: {len(q5_borough_age_activity_observed_export):,}"
)

print(
    "\nSaved borough-age-activity complete panel:"
)
print(
    borough_age_activity_complete_path
)
print(
    f"Rows: {len(q5_borough_age_activity_complete_panel):,}"
)

print(
    "\nSaved borough-age observed panel:"
)
print(
    borough_age_observed_path
)
print(
    f"Rows: {len(q5_borough_age_observed_export):,}"
)

print(
    "\nSaved borough-age complete panel:"
)
print(
    borough_age_complete_path
)
print(
    f"Rows: {len(q5_borough_age_complete_panel):,}"
)

print(
    "\nSaved borough-activity observed panel:"
)
print(
    borough_activity_observed_path
)
print(
    f"Rows: {len(q5_borough_activity_observed_export):,}"
)

print(
    "\nSaved borough-activity complete panel:"
)
print(
    borough_activity_complete_path
)
print(
    f"Rows: {len(q5_borough_activity_complete_panel):,}"
)

print(
    "\nSaved dataset diagnostics:"
)

for output_path in [
    sample_summary_output_path,
    panel_quality_output_path,
    year_quality_output_path,
    series_coverage_output_path,
]:
    print(
        output_path
    )

Saved respondent-level dataset:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_respondent_master_with_variables.csv
Rows: 96,629

Saved borough-age-activity observed panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_versionA_borough_age_activitylevel_panel_observed.csv
Rows: 4,290

Saved borough-age-activity complete panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_versionA_borough_age_activitylevel_panel_complete.csv
Rows: 4,608

Saved borough-age observed panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_versionA_borough_age_panel_observed.csv
Rows: 1,523

Saved borough-age complete panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_versionA_borough_age_panel_complete.csv
Rows: 1,536

Saved borough-activity observed panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_outputs/q5_versionA_borough_activitylevel_panel_observed.csv
Rows: 576

Saved borough-activity complete panel:
/Users/zsh/Downloads/PROJECT/code/q4_dataset_out